# **Pre-Processing data**

In [1]:
import os
import py_vncorenlp
import json
import re
from tqdm.auto import tqdm

c:\Users\Admin\anaconda3\envs\datascience\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NOTEBOOK_DIR = os.getcwd()
CAPSTONE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASET_DIR = os.path.join(CAPSTONE_DIR, "data")
MODEL_DIR = "D:/VnCoreNLP" # This is belong to your file location 

## Build the article corpus

In [3]:
# model = py_vncorenlp.VnCoreNLP(save_dir= MODEL_DIR)

In [4]:
# with open(os.path.join(DATASET_DIR, "stopwords_processed.txt"), "r", encoding="utf-8") as f:
#     stopwords_list = list(map(str.strip, f))

# pattern = r"\b(" + "|".join(map(re.escape, stopwords_list)) + r")\b"
# print(stopwords_list)

In [5]:
mapping_urls_index = {}

input_file = os.path.join(DATASET_DIR, "Indexed_cleanest_data.json")
with open(input_file, "r", encoding="utf-8") as json_file:
    data = json.load(json_file)

for article in data:
    mapping_urls_index[article["url"]] = article["article_index"]


In [ ]:
input_file = os.path.join(DATASET_DIR, "Indexed_cleanest_data.json")
output_file = os.path.join(DATASET_DIR, "processed_articles_corpus.json")

with open(input_file, "r", encoding="utf-8") as json_file:
    data = json.load(json_file)

total_articles = len(data)

with open(output_file, "w", encoding="utf-8") as out_file:
    out_file.write("[\n")   
    
    for i, article in enumerate(tqdm(data, total=total_articles, desc="Processing documents")):
        processed_articles = {}
        article_title = article["Title"]
        # article_title_split = article_title.split(". ")
        # if len(article_title_split) > 1:
        #     article_title = article_title_split[1]
        # else:
        #     article_title = ""
        article_desc = article["Description"]
        content = article_title + ". " + article_desc
        article_list = model.word_segment(content)
        article_segment_only = " ".join(article_list)

        processed_articles["article_index"] = article["article_index"]
        processed_articles["url"] = article["url"]
        processed_articles["processed_title_description"] = re.sub(pattern, "", article_segment_only)
        processed_articles["processed_title_description"] = re.sub(r"\s+", " ", processed_articles["processed_title_description"]).strip()
        processed_articles["processed_title_description"] = re.sub(r'"', '', processed_articles["processed_title_description"])

        processed_articles["related_index"] = [mapping_urls_index[url] for url in article["Related_link"]]
        
        json.dump(processed_articles, out_file, ensure_ascii=False, indent=4)
        
        if i < total_articles - 1:
            out_file.write(",\n")
    
    out_file.write("\n]")

In [7]:
working_file = os.path.join(DATASET_DIR, "processed_articles_corpus.json")

with open(working_file, "r", encoding="utf-8") as f:
    data = json.load(f)

total_articles = len(data)
print(total_articles)

for article in tqdm(data, total=total_articles, desc="fill null"):
    src_idx = article["article_index"]

    for target_idx in article["related_index"]:
        target_idx = int(target_idx)

        # check index hợp lệ
        if 1 <= target_idx <= total_articles:
            data[target_idx - 1]["related_index"].append(src_idx)
            data[target_idx - 1]["related_index"] = list(set(data[target_idx - 1]["related_index"]))

# with open(working_file, "w", encoding="utf-8") as f:
#     json.dump(data, f, ensure_ascii=False, indent=4)


25457


fill null: 100%|██████████| 25457/25457 [00:00<00:00, 385540.90it/s]


In [8]:
input_file = os.path.join(DATASET_DIR, "processed_articles_corpus.json")
output_file = os.path.join(DATASET_DIR, "sorted_processed_articles_corpus.json")

with open(input_file, "r", encoding="utf-8") as json_file:
    data = json.load(json_file)

total_articles = len(data)
print(total_articles)

sorted_data = []

for article in tqdm(data, total=total_articles, desc="sort null"):
    if len(article["related_index"]) > 0:
        sorted_data.append(article)

print(len(sorted_data))

# with open(output_file, "w", encoding="utf-8") as f:
#     json.dump(sorted_data, f, ensure_ascii=False, indent=4)

25457


sort null: 100%|██████████| 25457/25457 [00:00<00:00, 2952586.81it/s]

23182


## Train test split with no data leakage

In [9]:
# Sorting article with related_link is not non

In [12]:
from collections import defaultdict

adj = defaultdict(set)

for d in data:
    i = d["article_index"]
    for j in d["related_index"]:
        adj[i].add(j)
        adj[j].add(i)   # đảm bảo graph vô hướng


In [ ]:
import random
index = []
for data_point in data:
    index.append(data_point["article_index"])

all_indices = set(index)
train_set = set()

while len(train_set) < 16227:
    seed = random.choice(list(all_indices - train_set))
    queue = [seed]

    while queue:
        u = queue.pop()
        if u in train_set:
            continue
        train_set.add(u)
        queue.extend(adj[u])


In [14]:
test_set = all_indices - train_set


In [20]:
train_list = list(train_set)
len(train_list )

16227

In [21]:
test_list = list(test_set)
len(test_list)

6955

In [22]:
train_list[:10]

[1, 3, 7, 8, 9, 10, 11, 14, 16, 19]

In [24]:
train_data = []
test_data = []

for data_point in data:
    if data_point["article_index"] in train_list:
        train_data.append(data_point)
    else:
        test_data.append(data_point)

print(len(train_data))
print(len(test_data))

# Lưu file train
with open(os.path.join(DATASET_DIR, "train_sorted_processed_articles_corpus.json"), 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=4)

# Lưu file test
with open(os.path.join(DATASET_DIR, "test_sorted_processed_articles_corpus.json"), 'w', encoding='utf-8') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=4)

print(f"Tổng số mẫu: {len(data)}")
print(f"Số mẫu train: {len(train_data)}")
print(f"Số mẫu test: {len(test_data)}")

16227
6955
Tổng số mẫu: 23182
Số mẫu train: 16227
Số mẫu test: 6955
